# 08_score_testfp_full — chấm 16 checkpoint trên bài báo NGUYÊN VẸN

**Chạy trên Kaggle** (cần GPU + trọng số model).

## Đo cái gì

`test_fp` cũ lấy 3 mảnh/bài (~11% một bài thật), nên tỉ lệ oan mức bài đo được (5,2%) là
**giới hạn dưới**. Bộ `test_fp_full` giữ MỌI mảnh của bài → k ≈ 22,6 mảnh/bài.

Câu hỏi: **ở độ dài bài THẬT thì tỉ lệ oan là bao nhiêu?**

Hai sàn (GBM bề mặt + TF-IDF) đã đo trước trên CPU: **20,3% – 28,6%**, tức nở **12,4–12,7×**
so với mức mảnh. Notebook này lấy con số đó cho 16 model thật.

## Bộ dữ liệu — vì sao nó sạch

| | |
|---|---|
| 1.200 bài · 27.119 mảnh | trung vị 19 mảnh/bài, TB 22,6 |
| `paper_id` **không** nằm trong train / val / test_ood_model / test_fp | 5.274 id đã loại |
| Xuất bản **2010–2015** | trước ChatGPT 7 năm → nhãn human **chắc chắn**, không suy đoán |
| Cắt bằng **đúng** `split_into_sentence_chunks` của `get-dataset.ipynb` | chỉ bỏ bước `random.sample(k=3)` |
| Đã `canonicalize` | khớp bản model được train |
| **Chỉ có lớp human** | nên notebook này đo OAN, KHÔNG đo bắt được AI |

## Trình tự
```
§1 registry + tìm artifact   -> thiếu cái nào thì in link cái đó
§2 GUARD                     -> checkpoint có đúng là bản deconf2 không
§3 nạp dữ liệu + predict_proba (bản sao nguyên văn từ train_ddp.py)
§4 chấm điểm  -> pred_test_fp_full_<artifact>.parquet, bỏ qua file đã có
§5 tỉ lệ oan mức mảnh vs mức bài, theo từng nhóm k
```

> **Ước chi phí**: 27.119 dòng ≈ 20 phút/model trên dual-T4 → **16 model ≈ 5–6 giờ**.
> Hết session thì chạy lại, §4 tự bỏ qua file đã chấm. Muốn nhẹ hơn: đặt `MAX_ROWS`.

## 1. Registry + tìm artifact

In [ ]:
import gc, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

BASE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'preprocess.py').exists()), Path.cwd())
OUT = (Path('/kaggle/working') if Path('/kaggle/working').exists() else BASE / 'artifacts') / 'pred_testfp_full'
OUT.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [BASE, BASE / 'artifacts', Path('/kaggle/working'), Path('/kaggle/input')]
DATA_NAME = 'dataset_test_fp_full_deconf2.parquet'

MAX_LEN  = 512     # PHẢI khớp lúc train — đổi là số không so được với val/test_fp nữa
FP16     = False   # True nhanh ~2x nhưng lệch chữ số cuối; False cho khớp train
MAX_ROWS = None    # đặt số (vd 3000) để chạy THỬ nhanh; None = chấm hết
# Chạy thử 1 model: đặt ONLY = ['artifacts_deberta_v3_large_seed42'] (tên phải khớp REGISTRY).
# Kết hợp với MAX_ROWS=3000 thì một lượt chỉ ~2 phút, đủ để biết pipeline có chạy không.
ONLY     = None    # None = chấm hết 16 artifact

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPU  = torch.cuda.device_count() if DEVICE == 'cuda' else 0
BATCH  = 32

# (nhánh, tên model, seed, link notebook đã chạy)
REGISTRY = {
 'artifacts_roberta_large_seed42':            ('supervised','roberta_large',42,'https://www.kaggle.com/code/<kaggle-user>/roberta-supervised'),
 'artifacts_roberta_large_seed43':            ('supervised','roberta_large',43,'https://www.kaggle.com/code/<kaggle-user>/roberta-supervised'),
 'artifacts_roberta_large_seed44':            ('supervised','roberta_large',44,'https://www.kaggle.com/code/<kaggle-user>/roberta-supervised'),
 'artifacts_deberta_v3_large_seed42':         ('supervised','deberta_v3_large',42,'https://www.kaggle.com/code/<kaggle-user>/superviseddetector-debertav3'),
 'artifacts_deberta_v3_large_seed43':         ('supervised','deberta_v3_large',43,'https://www.kaggle.com/code/<kaggle-user>/superviseddetector-debertav3'),
 'artifacts_deberta_v3_large_seed44':         ('supervised','deberta_v3_large',44,'https://www.kaggle.com/code/<kaggle-user>/superviseddetector-debertav3'),
 'artifacts_modernbert_large_seed42':         ('supervised','modernbert_large',42,'https://www.kaggle.com/code/doquangluccqh/modernbert-supervised'),
 'artifacts_modernbert_large_seed43':         ('supervised','modernbert_large',43,'https://www.kaggle.com/code/doquangluccqh/modernbert-supervised'),
 'artifacts_modernbert_large_seed44':         ('supervised','modernbert_large',44,'https://www.kaggle.com/code/doquangluccqh/modernbert-supervised'),
 'artifacts_xlm_roberta_large_seed42':        ('supervised','xlm_roberta_large',42,'https://www.kaggle.com/code/mhien01604/xlm-roberta-large'),
 'artifacts_xlm_roberta_large_seed43':        ('supervised','xlm_roberta_large',43,'https://www.kaggle.com/code/mhien01604/xlm-roberta-large'),
 'artifacts_xlm_roberta_large_seed44':        ('supervised','xlm_roberta_large',44,'https://www.kaggle.com/code/mhien01604/xlm-roberta-large'),
 'artifacts_greater_roberta_large_seed42':    ('adversarial','GREATER+roberta-large',42,'https://www.kaggle.com/code/variphx/greater-roberta'),
 'artifacts_greater_xlm_roberta_large_seed42':('adversarial','GREATER+xlm-roberta-large',42,'https://www.kaggle.com/code/<kaggle-user>/xlm-al-greater'),
 'artifacts_radar_seed42':                    ('adversarial','RADAR+roberta-large',42,'https://www.kaggle.com/code/thoinhcng/radar-roberta'),
 'artifacts_radar_xlm_seed42':                ('adversarial','RADAR+xlm-roberta-large',42,'https://www.kaggle.com/code/doquangluccqh/xlm-al-radar'),
}


def find_file(name):
    for root in SEARCH_ROOTS:
        if root.exists():
            for hit in root.rglob(name):
                if hit.is_file():
                    return hit
    return None


def find_art(name):
    for root in SEARCH_ROOTS:
        if root.exists():
            for hit in root.rglob(name):
                if hit.is_dir() and (hit / 'model').exists():
                    return hit
    return None


DATA_PATH = find_file(DATA_NAME)
assert DATA_PATH, (f'Không thấy {DATA_NAME}.\n'
                   'Trên Kaggle: Add Data -> dataset chứa file này (chạy build_testfp_full.py ở local để tạo).')
print(f'DATA_PATH = {DATA_PATH}')
print(f'DEVICE = {DEVICE} | {N_GPU} GPU | BATCH/GPU = {BATCH}')

found = {a: find_art(a) for a in REGISTRY}
ok = {a: p for a, p in found.items() if p}
print(f'\n{len(ok)}/{len(REGISTRY)} artifact có model/\n')
for a, (br, nm, sd, url) in REGISTRY.items():
    print(f"  {'OK  ' if found[a] else 'MISS'} {nm:28} seed{sd}  {found[a] or url}")

## 2. GUARD — checkpoint có đúng là bản `deconf2` không

Hai chỗ hỏng trong im lặng mà §4 không bắt được:

1. `if dst.exists(): continue` — Add Data từ output của lần chạy TRƯỚC (checkpoint tiền-deconf2)
   thì nó bỏ qua model đó và tái dùng score CŨ, in "đã có" rồi đi tiếp, không báo lỗi.
2. Không có gì trong notebook này biết checkpoint đến từ biến thể dữ liệu nào.

Dấu vân tay: số dòng `val` trong `metrics.json` — `deconf2` → 876, bản cũ → 1030.

In [ ]:
EXPECT_VAL_N = 876
bad = []
print(f"{'artifact':46}{'val n':>7}{'tau':>10}  trạng thái")
print('-' * 84)
for a in sorted(ok):
    art = found[a]; n = tau = None
    mp = art / 'metrics.json'
    if mp.exists():
        m = json.loads(mp.read_text()); tau = m.get('tau')
        n = next((r.get('n') for r in (m.get('metrics') or [])
                  if str(r.get('split', '')).strip() == 'val'), None)
    st = ('deconf2 OK' if n == EXPECT_VAL_N else
          'THIẾU metrics.json/val n' if n is None else
          f'!! KHÔNG phải deconf2 (mong {EXPECT_VAL_N})')
    if n is not None and n != EXPECT_VAL_N:
        bad.append(a)
    if (OUT / f'pred_test_fp_full_{a}.parquet').exists():
        st += '  · ĐÃ CÓ pred -> §4 sẽ BỎ QUA'
    print(f"{a:46}{str(n if n is not None else '?'):>7}"
          f"{(f'{tau:.4f}' if isinstance(tau, (int, float)) else '?'):>10}  {st}")

assert not bad, f'\nCheckpoint KHÔNG phải deconf2: {bad}\nGỡ chúng khỏi Add Data rồi chạy lại.'
print('\nSạch: mọi checkpoint là deconf2.')

## 3. Nạp dữ liệu + `predict_proba`

`score_on` / `predict_proba` là **bản sao nguyên văn** từ `train_ddp.py` (qua `07_score_mix.ipynb`).
Đừng viết lại — khác một chi tiết (`truncation`, `pad_to_multiple_of=8`, lấy `[:, 1]`) là score
không so được với `val` / `test_fp` trong báo cáo nữa.

In [ ]:
df = pd.read_parquet(DATA_PATH)
if MAX_ROWS:
    df = df.head(MAX_ROWS)
df['label'] = 0                     # bộ này chỉ có lớp human
k = df.groupby('paper_id').size()
print(f'{len(df):,} mảnh · {df.paper_id.nunique():,} bài')
print(f'  mảnh/bài: min {k.min()} · trung vị {k.median():.0f} · TB {k.mean():.2f} · max {k.max()}')
print(f'  từ/mảnh : trung vị {df.human_text.str.split().str.len().median():.0f}')
TEXTS = df['human_text'].tolist()
KEEP = ['sample_id', 'paper_id', 'source', 'label']


@torch.no_grad()
def score_on(model, tokenizer, texts, dev):
    """Chấm `texts` bằng `model` đã nằm trên `dev`. Lõi giống train_ddp.py."""
    model.eval(); out = []
    for i in range(0, len(texts), BATCH):
        enc = tokenizer(texts[i:i + BATCH], truncation=True, max_length=MAX_LEN,
                        padding=True, pad_to_multiple_of=8, return_tensors='pt').to(dev)
        with torch.autocast('cuda', enabled=FP16 and str(dev).startswith('cuda')):
            logits = model(**enc).logits
        out.extend(torch.softmax(logits.float(), -1)[:, 1].cpu().tolist())
    return out


def predict_proba(art_dir, texts):
    """Nạp model rồi chấm. N_GPU>1 thì nạp 1 bản/GPU và chạy song song bằng thread.

    Chia theo BƯỚC NHẢY texts[i::N] chứ không cắt nửa đầu/nửa sau: ghép lại chỉ cần
    res[i::N] = shard_i, và tải giữa các GPU cân nhau kể cả khi độ dài văn bản không đều.
    """
    tok = AutoTokenizer.from_pretrained(art_dir / 'model')
    devs = [f'cuda:{i}' for i in range(N_GPU)] if N_GPU >= 1 else ['cpu']
    models = [AutoModelForSequenceClassification.from_pretrained(art_dir / 'model').to(d).eval()
              for d in devs]

    if len(devs) == 1:
        res = score_on(models[0], tok, texts, devs[0])
    else:
        from concurrent.futures import ThreadPoolExecutor
        res = [None] * len(texts)
        shards = [texts[i::len(devs)] for i in range(len(devs))]
        with ThreadPoolExecutor(max_workers=len(devs)) as ex:
            parts = list(ex.map(lambda a: score_on(a[0], tok, a[1], a[2]),
                                list(zip(models, shards, devs))))
        for i, p in enumerate(parts):
            res[i::len(devs)] = p

    for m in models:
        del m
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return np.array(res)

## 4. Chấm điểm — mỗi model một file, bỏ qua file đã có

In [ ]:
todo = {a: p for a, p in ok.items() if not ONLY or a in ONLY}
if ONLY:
    missing = [a for a in ONLY if a not in ok]
    assert not missing, f'ONLY có tên không tìm thấy artifact: {missing}'
    print(f'CHỈ chấm {len(todo)}/{len(ok)} artifact (ONLY)')

for a, art in todo.items():
    dst = OUT / f'pred_test_fp_full_{a}.parquet'
    if dst.exists():
        print(f'SKIP {a} — đã có'); continue
    br, nm, sd, _ = REGISTRY[a]
    t0 = time.time()
    sc = predict_proba(art, TEXTS)

    out = df[KEEP].copy()
    out['artifact'], out['branch'], out['model'], out['seed'] = a, br, nm, sd
    out['score'] = sc
    out.to_parquet(dst, index=False)
    print(f'OK  {nm:28} seed{sd}  {time.time() - t0:7.1f}s  '
          f'score mean {sc.mean():.4f} std {sc.std():.4f}')
print(f'\n-> {OUT}')

## 5. Tỉ lệ oan: mức mảnh vs mức bài

τ lấy từ `metrics.json` của chính artifact đó — **đúng ngưỡng đã chốt trên `val` ở FPR=1%**,
không hiệu chuẩn lại. Đây là câu hỏi *"ngưỡng đang deploy gây oan bao nhiêu ở bài thật"*.

Cột `nếu độc lập` = `1-(1-p)^k`. Hai sàn đo trước cho thấy công thức **phóng đại ở k lớn**
(sai +66% ở k≈43), nên cột `đo được` mới là con số để báo cáo.

In [ ]:
rows = []
for f in sorted(OUT.glob('pred_test_fp_full_*.parquet')):
    d = pd.read_parquet(f)
    a = d.artifact.iloc[0]
    mp = found.get(a, Path('.')) / 'metrics.json'
    if not mp.exists():
        print(f'bỏ {a}: thiếu metrics.json -> không có tau'); continue
    tau = json.loads(mp.read_text())['tau']
    d['co'] = d.score >= tau
    g = d.groupby('paper_id')
    p = float(d.co.mean()); kk = g.size()
    rows.append({'model': f'{d.model.iloc[0]}_s{d.seed.iloc[0]}',
                 'tau': tau,
                 'oan mức mảnh': p,
                 'oan mức bài': float(g.co.any().mean()),
                 'nếu độc lập': float((1 - (1 - p) ** kk).mean()),
                 'k TB': float(kk.mean()), 'n bài': int(len(kk))})

T = pd.DataFrame(rows).sort_values('oan mức bài')
T['số lần'] = (T['oan mức bài'] / T['oan mức mảnh']).round(1)
print(T.round(4).to_string(index=False))
T.to_csv(OUT / 'summary_test_fp_full.csv', index=False)

print('\nMốc so sánh (đã đo trên CPU, cùng 1.200 bài này):')
print('  sàn BỀ MẶT  : oan mảnh 2,25% -> oan bài 28,6%  (12,7x)')
print('  sàn TỪ VỰNG : oan mảnh 1,63% -> oan bài 20,3%  (12,4x)')
print('  test_fp CŨ (k=2,81) : nở 2,4-2,8x trên 20/20 detector')